# Griffin — Quick Start

Import the `src` package instead of defining functions inline.

Run this setup cell once, then jump to any step directly.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import glob

# Import everything from the src package
from src import (
    load_image, load_lidar, load_pose_griffin, load_calib_griffin,
    load_labels_for_frame, get_file_lists,
    project_lidar_to_image, project_ego_to_img, ego_box_corners_3d,
    plot_surround_cameras, plot_bev, plot_front_view,
    plot_fusion, plot_bev_with_boxes, plot_boxes_on_image,
    CAT_COLORS,
)

plt.rcParams['figure.dpi'] = 120
print('Imports OK')

In [ ]:
# ── Set your dataset path here ────────────────────────────────────────────
DATASET_ROOT = '../datasets/griffin_50scenes_25m/griffin-release/griffin_50scenes_25m/griffin-release'
VEH   = os.path.join(DATASET_ROOT, 'vehicle-side')
DRONE = os.path.join(DATASET_ROOT, 'drone-side')

# Build all file lists at once
files = get_file_lists(VEH, DRONE)

CALIB_DIR = os.path.join(VEH, 'calib')

print(f'Vehicle front frames : {len(files["veh_fronts"])}')
print(f'LiDAR frames         : {len(files["lidar_plys"])}')
print(f'Pose files           : {len(files["pose_files"])}')
print(f'Label files          : {len(files["label_files"])}')

In [ ]:
# ── Config — change these two lines to explore ────────────────────────────
CAMERA    = 'front'
FRAME_IDX = 700

# Load data for this frame
pts                    = load_lidar(files['lidar_plys'][FRAME_IDX])
T_ENU_to_ego, pose     = load_pose_griffin(files['pose_files'][FRAME_IDX])
K, T_ego_to_sensor     = load_calib_griffin(CALIB_DIR, camera=CAMERA)
anns                   = load_labels_for_frame(VEH, files['pose_files'], FRAME_IDX)

cam_imgs = sorted(glob.glob(os.path.join(VEH, 'camera', CAMERA, '*.png')))
img      = load_image(cam_imgs[FRAME_IDX])
h, w     = img.shape[:2]

uvd = project_lidar_to_image(pts[:,:3], K, T_ego_to_sensor, T_ENU_to_ego, h, w)

print(f'Camera: {CAMERA}  |  Frame: {FRAME_IDX}')
print(f'LiDAR pts: {len(pts):,}  |  Projected: {len(uvd):,}')
print(f'Annotations: {len(anns)}')

In [ ]:
# Sensor fusion overlay
fig = plot_fusion(img, uvd, camera=CAMERA)
plt.show()

In [ ]:
# BEV with annotations
fig = plot_bev_with_boxes(pts, anns, pose['x'], pose['y'])
plt.show()

In [ ]:
# 3D boxes on camera image
fig = plot_boxes_on_image(img, anns, K, T_ego_to_sensor, camera=CAMERA)
plt.show()